In [32]:
import re   
import math
from collections import Counter

In [33]:
dataset = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM"),
]


In [34]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]','',text)
    return text.split()

In [35]:
word_counts = {
    "SPAM": Counter(),
    "HAM": Counter (),
}

doc_counts = {
    "SPAM": 0,
    "HAM": 0
}

for text, label in dataset:
    doc_counts[label] += 1
    words = preprocess(text)
    word_counts[label].update(words)

In [36]:
total_words = {
    label: sum(word_counts[label].values())
    for label in word_counts
}

# Vocabulary
vocab = set()
for label in word_counts:
    vocab.update(word_counts[label].keys())

V = len(vocab)

print("Document Counts:", doc_counts)
print("Total Words per Class:", total_words)
print("Vocabulary Size:", V)

Document Counts: {'SPAM': 5, 'HAM': 6}
Total Words per Class: {'SPAM': 22, 'HAM': 33}
Vocabulary Size: 44


In [37]:
print("=== BAG OF WORDS ===\n")

for label in ["SPAM", "HAM"]:
    print(f"{label} WORD FREQUENCIES:\n")
    
    # Sort words alphabetically for clean output
    for word, count in sorted(word_counts[label].items()):
        print(f"{word:10} : {count}")
    
    print("\nTotal words in", label, "=", total_words[label])
    print("-" * 40)


=== BAG OF WORDS ===

SPAM WORD FREQUENCIES:

50         : 1
a          : 1
click      : 1
for        : 2
free       : 2
get        : 1
here       : 1
iphone     : 1
limited    : 1
lowest     : 1
meds       : 1
money      : 1
now        : 1
off        : 1
price      : 1
prizes     : 1
time       : 1
today      : 1
win        : 1
your       : 1

Total words in SPAM = 22
----------------------------------------
HAM WORD FREQUENCIES:

3          : 1
are        : 2
at         : 2
can        : 1
catch      : 1
dinner     : 1
for        : 1
hi         : 1
how        : 1
in         : 1
lets       : 1
meeting    : 2
mom        : 1
office     : 2
on         : 1
pm         : 1
report     : 1
send       : 1
still      : 1
team       : 1
the        : 3
tomorrow   : 2
up         : 1
we         : 1
you        : 2

Total words in HAM = 33
----------------------------------------


In [38]:
total_docs = len(dataset)
priors = {
    label: doc_counts[label] / total_docs
    for label in doc_counts
}

print("Priors:", priors)
print("Vocabulary size:", V)
print()

Priors: {'SPAM': 0.45454545454545453, 'HAM': 0.5454545454545454}
Vocabulary size: 44



In [39]:
def likelihood(word, label):
    # Laplace smoothing
    return (word_counts[label][word] + 1) / (total_words[label] + V)

In [40]:
def predict(text):
    words = preprocess(text)
    scores = {}

    for label in ["SPAM", "HAM"]:
        prob = priors[label]
        
        for word in words:
            prob *= likelihood(word, label)
        
        scores[label] = prob    
        prediction = max(scores, key=scores.get)

    return prediction, scores

In [41]:
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

for sentence in test_sentences:
    prediction, scores = predict(sentence)
    
    print("Sentence:", sentence)
    print("Prediction:", prediction)
    print("Probabilities:", scores)


Sentence: Limited offer, click here!
Prediction: SPAM
Probabilities: {'SPAM': 1.9164238366023312e-07, 'HAM': 1.551656783987893e-08}
Sentence: Meeting at 2 PM with the manager.
Prediction: HAM
Probabilities: {'SPAM': 8.332393479397675e-14, 'HAM': 2.4471240512104998e-12}
